# Day 3

## Read and Parse Data

In [2]:
with open('input.txt') as f:
    data = f.read().split('\n')

In [3]:
data

['...............776..............552........968..................589...26...........484..............958......186....546.........484.........',
 '.........*.........778....................*....124...................................*...............*........%..26.........................',
 '......194..380....@....900..........639....467........478*..............582...........798.............326...........894.........#...........',
 '904...........*2.......#......259.....*..........801......464................597.569.............794+................$..218....502..........',
 '...*.....................-...$.....%..431.........*...810.....840+.668..........*.......144=.............................../...........%627.',
 '...890...497.........829.643....504..........465..502..............*........488...................787.184...601....215........-450..........',
 '.......................*............684.....*.............466...970....646..%...........399...........*.............=....88.....

# Part 1

In [6]:
# List of possible digits 0-9 as strings
digits = list(map(str, range(10)))

# Set of possible directions (include diagonal)
directions = {
    (0, 1), (0, -1), (1, 0), (-1, 0), 
    (1, 1), (1, -1), (-1, 1), (-1, -1)
    }

In [10]:
def get_number_to_end(data, pos):
    number = ''
    idxs = set()
    n = len(data[0])
    i, j = pos
    idxs.add((i, j))
    
    while 0 <= j < n and data[i][j] in digits:
        number += data[i][j]
        idxs.add((i, j))
        j += 1

    return number, idxs

In [26]:
def check_in_bounds(pos, shape):
    i, j = pos
    m, n = shape

    if (0 <= i < m) and (0 <= j < n):
        return True
    else:
        return False

In [32]:
def check_around(data, idxs, bounds):
    # Check every digit of the number
    for idx in idxs:
        i, j = idx
        # Check every direction
        # Note 1.
        for direction in directions:
            nx = i + direction[0]
            ny = j + direction[1]

            # If out-of-bounds check next direction
            if check_in_bounds((nx, ny), bounds):
                # If inbounds and is a symbol --> valid number
                if data[nx][ny] != '.' and data[nx][ny] not in digits:
                    return True
    # If no symbol was found around the number --> Not valid
    return False

Note 1:

This has redundant checks:
Top-right of first digit is the same as top of second digit and top-left of third
It will be slower but should not be significant

In [34]:
# Data shape (array like)
m = len(data)
n = len(data[0])
# 140x140

sum = 0
seen_nums_idx = set()

# Runs top to bot and left to right
# The first digit encountered will be the first of that number
for i in range(m):
    for j in range(n):
        # If already checked that number, continue to next iteration
        if (i,j) in seen_nums_idx:
            continue
        
        if data[i][j] in digits:
            number, idxs = get_number_to_end(data, (i, j))
            if check_around(data=data, idxs=idxs, bounds=(m,n)):
                seen_nums_idx = seen_nums_idx.union(idxs)
                sum += int(number)

sum

539637

# Part 2

In [35]:
def get_whole_number(data, pos, bounds):

    i, j = pos

    # i doesnt change, a number has all its digits in the same row
    # Find the first digit (min j)
    while data[i][j] in digits and check_in_bounds((i, j), bounds):
        j -= 1

    # Once we are out of that loop it means we are out of bounds or 
    # found a non digit character
    j += 1

    return get_number_to_end(data, (i, j))

In [37]:
def check_gear(data, pos):
    shape = (len(data), len(data[0]))
    i, j = pos
    seen_idxs = set()
    neigh_num = 0
    current_gear_ratio = 1
    for direction in directions:
        nx = i + direction[0]
        ny = j + direction[1]

        if (nx, ny) in seen_idxs:
            continue

        if check_in_bounds(pos=(nx, ny), shape=shape):
            if data[nx][ny] in digits:
                num, idxs = get_whole_number(data=data, pos=(nx, ny), bounds=shape)
                seen_idxs = seen_idxs.union(idxs)
                current_gear_ratio *= int(num)
                neigh_num += 1

    if neigh_num == 2:
        return current_gear_ratio
    
    else:
        return 0

In [38]:
m, n = len(data), len(data[0])
gear_ratio = 0

for i in range(m):
    for j in range(n):
        if data[i][j] == '*':
            gear_ratio += check_gear(data, (i, j))

gear_ratio

82818007